# Pandas - zadania 31-40

Zakres: łączenie tabel, daty, zapytania w stylu SQL i studium przypadku (sekcje 8–10).

Rozwiązuj zadania w pustych komórkach z komentarzem `# tutaj wpisz rozwiązanie`. Gotowe komórki z danymi uruchom bez zmian.

### Import bibliotek i dane

Dane pobieramy z internetu (wymagane połączenie). Funkcja `wczytaj_csv_z_internetu` ma dwie wersje – wybierz wersję dla swojego systemu.

In [ ]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

URL = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/'

Uruchom **tylko jedną** z dwóch poniższych komórek – zależnie od systemu:

**Wersja dla Windows** – uruchom tę komórkę tylko na Windowsie (bezpośrednie `pd.read_csv(url)`):

In [ ]:
def wczytaj_csv_z_internetu(url, **kwargs):
    return pd.read_csv(url, **kwargs)

**Wersja dla macOS i Linux** – uruchom tę komórkę tylko na macOS lub Linuksie. Pobiera dane biblioteką `requests`, bo `pd.read_csv(url)` potrafi na macOS zgłosić błąd `CERTIFICATE_VERIFY_FAILED`. Ta wersja działa też na Windowsie, więc w razie kłopotów możesz jej użyć wszędzie:

In [ ]:
def wczytaj_csv_z_internetu(url, **kwargs):
    odpowiedz = requests.get(url, timeout=30)
    odpowiedz.raise_for_status()
    return pd.read_csv(StringIO(odpowiedz.text), **kwargs)

### Ćwiczenie 31
Tabele `sklep_a` i `sklep_b` zawierają sprzedaż w dwóch sklepach.

a) Sklej je pod sobą w jedną tabelę z nowym, ciągłym indeksem.

b) Sklej je tak, aby było widać, z którego sklepu pochodzi wiersz (dodatkowy poziom indeksu `'A'` i `'B'`).

__Oczekiwany wynik:__

```
  produkt  sprzedaż
0   chleb       120
1   mleko       200
2     ser        60
3   masło        80
4   jajka       150
```

```
    produkt  sprzedaż
A 0   chleb       120
  1   mleko       200
B 0     ser        60
  1   masło        80
  2   jajka       150
```

__Wskazówka:__ Użyj `pd.concat()` z parametrami _ignore_index_ oraz _keys_.

In [ ]:
sklep_a = pd.DataFrame({'produkt': ['chleb', 'mleko'], 'sprzedaż': [120, 200]})
sklep_b = pd.DataFrame({'produkt': ['ser', 'masło', 'jajka'], 'sprzedaż': [60, 80, 150]})
sklep_a, sklep_b

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 32
Tabela `kwartał_2` ma dodatkową kolumnę `koszty`, której nie ma w tabeli `kwartał_1`. Sklej obie tabele pod sobą (z ciągłym indeksem), a powstałe braki zastąp zerami.

__Oczekiwany wynik:__

```
   miasto  przychód  koszty
0    Łódź       500     0.0
1  Gdańsk       420     0.0
2    Łódź       530   300.0
3  Gdańsk       450   280.0
```

__Wskazówka:__ Użyj `pd.concat()` i `fillna()`.

In [ ]:
kwartał_1 = pd.DataFrame({'miasto': ['Łódź', 'Gdańsk'], 'przychód': [500, 420]})
kwartał_2 = pd.DataFrame({'miasto': ['Łódź', 'Gdańsk'], 'przychód': [530, 450], 'koszty': [300, 280]})
kwartał_1, kwartał_2

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 33
Tabela `pracownicy` zawiera pracowników i identyfikator działu, a tabela `działy` – nazwy działów. W dziale `4` nie pracuje nikt, a pracownik Marek ma dział, którego nie ma w tabeli `działy` (dział `9`).

a) Złącz tabele tak, aby zostali **tylko** pracownicy mający dział w obu tabelach.

b) Złącz je tak, aby zostały **wszystkie** wiersze z obu tabel, i dodaj kolumnę pokazującą, z której tabeli pochodzi wiersz.

__Oczekiwany wynik:__

```
    imię  dział_id nazwa_działu
0   Anna         1     Sprzedaż
1  Piotr         2           IT
2    Ewa         1     Sprzedaż
```

```
    imię  dział_id nazwa_działu      _merge
0   Anna         1     Sprzedaż        both
1    Ewa         1     Sprzedaż        both
2  Piotr         2           IT        both
3    NaN         4   Księgowość  right_only
4  Marek         9          NaN   left_only
```

__Wskazówka:__ Użyj `pd.merge(..., on='dział_id', how=...)` z wartościami `'inner'` i `'outer'` oraz parametru _indicator=True_.

In [ ]:
pracownicy = pd.DataFrame({'imię': ['Anna', 'Piotr', 'Ewa', 'Marek'], 'dział_id': [1, 2, 1, 9]})
działy = pd.DataFrame({'dział_id': [1, 2, 4], 'nazwa_działu': ['Sprzedaż', 'IT', 'Księgowość']})
pracownicy, działy

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 34
Tabela `sprzedane` zawiera sprzedaże (kod produktu, ilość, cena sprzedaży), a `katalog` – cenę katalogową produktów. W obu tabelach jest kolumna `cena`, a klucz ma w każdej inną nazwę.

a) Złącz tabele (dołącz do sprzedaży dane z katalogu), nadając kolumnom `cena` przyrostki `_sprzedaż` i `_katalog`. Dodaj kolumnę `rabat` = cena katalogowa minus cena sprzedaży.

b) Policz łączny rabat (suma kolumny `rabat`) dla każdego produktu (kolumna `nazwa`).

__Oczekiwany wynik:__

```
  kod_produktu  ilość  cena_sprzedaż kod    nazwa  cena_katalog  rabat
0            A      1           3400   A   Laptop          3500    100
1            C      2            880   C  Monitor           900     20
2            C      1            900   C  Monitor           900      0
3            B      5             75   B     Mysz            80      5
```

```
nazwa
Laptop     100
Monitor     20
Mysz         5
Name: rabat, dtype: int64
```

__Wskazówka:__ Użyj `pd.merge()` z parametrami _left_on_, _right_on_ i _suffixes_, a potem `groupby()`.

In [ ]:
sprzedane = pd.DataFrame({'kod_produktu': ['A', 'C', 'C', 'B'], 'ilość': [1, 2, 1, 5], 'cena': [3400, 880, 900, 75]})
katalog = pd.DataFrame({'kod': ['A', 'B', 'C'], 'nazwa': ['Laptop', 'Mysz', 'Monitor'], 'cena': [3500, 80, 900]})
sprzedane, katalog

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 35
Seria `daty_tekst` zawiera daty w formacie polskim (`dzień.miesiąc.rok`). Zamień ją na daty i zbuduj tabelę `wynik` z kolumnami:
- `data` – data,
- `miesiąc` – numer miesiąca,
- `dzień_tyg` – nazwa dnia tygodnia,
- `do_końca_roku` – liczba dni do 31 grudnia 2024.

__Oczekiwany wynik:__

```
        data  miesiąc dzień_tyg  do_końca_roku
0 2024-01-15        1    Monday            351
1 2024-02-29        2  Thursday            306
2 2024-12-31       12   Tuesday              0
3 2024-07-05        7    Friday            179
```

__Wskazówka:__ Użyj `pd.to_datetime(..., format='%d.%m.%Y')`, akcesora `.dt` (`month`, `day_name()`, `days`) oraz odejmowania od `pd.Timestamp('2024-12-31')`.

In [ ]:
daty_tekst = pd.Series(['15.01.2024', '29.02.2024', '31.12.2024', '05.07.2024'])
daty_tekst

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 36
Seria `temperatura` zawiera dzienne pomiary od 1 stycznia 2024 przez 90 dni (indeks to daty).

a) Wyświetl statystyki opisowe (zaokrąglone do 1 miejsca) dla lutego 2024.

b) Wyznacz średnią temperaturę w każdym miesiącu (zaokrągloną do 1 miejsca).

__Oczekiwany wynik:__

```
count    29.0
mean     11.6
std       4.7
min       3.6
25%       9.0
50%      11.8
75%      13.9
max      19.8
Name: temperatura, dtype: float64
```

```
2024-01-31     9.3
2024-02-29    11.6
2024-03-31    10.9
Freq: ME, Name: temperatura, dtype: float64
```

__Wskazówka:__ Dla indeksu dat działa `loc['2024-02']`. Średnie miesięczne policz przez `resample('ME').mean()`.

In [ ]:
rng = np.random.default_rng(0)
temperatura = pd.Series(rng.normal(10, 5, 90).round(1),
                        index=pd.date_range('2024-01-01', periods=90, freq='D'),
                        name='temperatura')
temperatura.head()

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 37
Seria `sprzedaż` zawiera dzienną sprzedaż. Zbuduj tabelę `wynik` z kolumnami:
- `sprzedaż` – dane wejściowe,
- `średnia_3dni` – średnia krocząca z 3 dni (zaokrąglona do 1 miejsca),
- `wczoraj` – sprzedaż z poprzedniego dnia,
- `zmiana_%` – procentowa zmiana względem poprzedniego dnia (zaokrąglona do 1 miejsca).

__Oczekiwany wynik:__

```
            sprzedaż  średnia_3dni  wczoraj  zmiana_%
2024-05-01       100           NaN      NaN       NaN
2024-05-02       120           NaN    100.0      20.0
2024-05-03        90         103.3    120.0     -25.0
2024-05-04       150         120.0     90.0      66.7
2024-05-05       130         123.3    150.0     -13.3
2024-05-06       170         150.0    130.0      30.8
2024-05-07       160         153.3    170.0      -5.9
2024-05-08       200         176.7    160.0      25.0
```

__Wskazówka:__ Użyj `rolling(3).mean()`, `shift(1)` i `pct_change()` (wynik pomnóż przez 100).

In [ ]:
sprzedaż = pd.Series([100, 120, 90, 150, 130, 170, 160, 200], index=pd.date_range('2024-05-01', periods=8), name='sprzedaż')
sprzedaż

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 38
Tabela `filmy` zawiera 979 najlepiej ocenianych filmów z serwisu IMDb. Odpowiedz w stylu SQL na pytanie:

```sql
SELECT content_rating, COUNT(*) AS liczba, AVG(duration) AS średni_czas
FROM filmy
WHERE star_rating >= 8
GROUP BY content_rating
HAVING COUNT(*) >= 5
ORDER BY średni_czas DESC
```

Wynik zaokrąglij do 1 miejsca po przecinku.

__Oczekiwany wynik:__

```
                liczba  średni_czas
content_rating                     
PG-13               57        136.8
R                  157        130.8
NOT RATED           47        125.1
G                   19        121.2
PG                  50        120.2
APPROVED            28        112.9
UNRATED             24        109.5
PASSED               6        105.3
```

__Wskazówka:__ Połącz metody łańcuchem: `query()` (WHERE), `groupby().agg()` z nazwanymi agregacjami, ponownie `query()` (HAVING) i `sort_values()`.

In [ ]:
filmy = wczytaj_csv_z_internetu(URL + 'imdb_1000.csv')
filmy.head()

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 39 (studium przypadku: ceny używanych aut, część 1)
Plik `data.csv` (w tym samym folderze) zawiera ogłoszenia o sprzedaży używanych samochodów w Indiach (cena `Price` w lakhach rupii).

a) Wczytaj plik do tabeli `auta`, usuń zbędną kolumnę `Unnamed: 0` i wyświetl wymiary tabeli.

b) Wyświetl cztery kolumny z największą liczbą braków.

c) Kolumny `Engine` (np. `'998 CC'`) i `Power` (np. `'58.16 bhp'`, czasem `'null bhp'`) są tekstem. Utwórz liczbowe kolumny `Engine_cc` i `Power_bhp` (nieczytelne wartości mają stać się brakami) i wyświetl cztery kolumny `Engine`, `Engine_cc`, `Power`, `Power_bhp` dla pierwszych 5 wierszy.

__Oczekiwany wynik:__

```
(6019, 13)
```

```
New_Price    5195
Seats          42
Engine         36
Power          36
dtype: int64
```

```
    Engine  Engine_cc      Power  Power_bhp
0   998 CC      998.0  58.16 bhp      58.16
1  1582 CC     1582.0  126.2 bhp     126.20
2  1199 CC     1199.0   88.7 bhp      88.70
3  1248 CC     1248.0  88.76 bhp      88.76
4  1968 CC     1968.0  140.8 bhp     140.80
```

__Wskazówka:__ Użyj `pd.read_csv()`, `drop(columns=...)`, `isna().sum().sort_values(...)`, `str.replace()` i `pd.to_numeric(..., errors='coerce')`.

In [ ]:
# tutaj wpisz rozwiązanie

### Ćwiczenie 40 (studium przypadku: ceny używanych aut, część 2)
Poniższa komórka przygotowuje tabelę `auta` (wczytanie i czyszczenie z poprzedniego ćwiczenia). Uruchom ją bez zmian.

Następnie:

a) Dodaj kolumnę `wiek` = 2019 minus `Year` (dane pochodzą z 2019 roku).

b) Dla każdego rodzaju paliwa (`Fuel_Type`) policz liczbę ogłoszeń, średnią cenę i średni wiek. Posortuj malejąco według średniej ceny, wartości zaokrąglij do 2 miejsc.

c) Zbuduj tabelę przestawną ze średnią ceną: wiersze to `Fuel_Type`, kolumny to `Transmission` (zaokrąglij do 2 miejsc).

d) Wyświetl pięć najdroższych aut (kolumny `Name`, `Year`, `Fuel_Type`, `Price`).

__Oczekiwany wynik:__

```
                               Name  Year  wiek
0            Maruti Wagon R LXI CNG  2010     9
1  Hyundai Creta 1.6 CRDi SX Option  2015     4
2                      Honda Jazz V  2011     8
3                 Maruti Ertiga VDI  2012     7
4   Audi A4 New 2.0 TDI Multitronic  2013     6
```

```
           liczba  średnia_cena  średni_wiek
Fuel_Type                                   
Electric        2         12.88         5.50
Diesel       3205         12.84         5.26
Petrol       2746          5.70         6.10
CNG            56          3.52         5.00
LPG            10          2.49         8.20
```

```
Transmission  Automatic  Manual
Fuel_Type                      
CNG                 NaN    3.52
Diesel            24.58    6.64
Electric          12.88     NaN
LPG                 NaN    2.49
Petrol            11.26    4.11
```

```
                                             Name  Year Fuel_Type   Price
4079  Land Rover Range Rover 3.0 Diesel LWB Vogue  2017    Diesel  160.00
5781                   Lamborghini Gallardo Coupe  2011    Petrol  120.00
5919                       Jaguar F Type 5.0 V8 S  2015    Petrol  100.00
1505              Land Rover Range Rover Sport SE  2019    Diesel   97.07
1974                           BMW 7 Series 740Li  2018    Petrol   93.67
```

__Wskazówka:__ Użyj `groupby().agg()` z nazwanymi agregacjami, `pd.pivot_table()` oraz `nlargest()`.

In [ ]:
auta = pd.read_csv('data.csv').drop(columns='Unnamed: 0')
auta['Engine_cc'] = pd.to_numeric(auta['Engine'].str.replace(' CC', ''), errors='coerce')
auta['Power_bhp'] = pd.to_numeric(auta['Power'].str.replace(' bhp', ''), errors='coerce')
auta.head(3)

In [ ]:
# tutaj wpisz rozwiązanie